NNM estimators

In [ ]:
#import dbm.sqlite3
#import openpyxl
#import pandas as pd
#import numpy as np
#from io import StringIO
#import datetime as dt
#import econtools
#import geopy.distance as geo
#import pyreadstat
#import pyarrow
#import scipy
#import pyreadr
#import scipy.stats as stats
#import statsmodels
#import matplotlib.pyplot as plt
#import statsmodels.api as sm
#import seaborn as sns
#import os, shutil
#from pathlib import Path
#from sklearn.datasets import load_diabetes
#from sklearn.model_selection import train_test_split
#from sklearn.preprocessing import StandardScaler
#from sklearn.linear_model import LogisticRegression
#from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

from datetime import date
print("Today's date is:",date.today())

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from statsmodels.formula.api import glm
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Load the data
# Step 1: Set the working directory (No direct equivalent, specify path)
data_path = "/Causal_Inference/Treatment_effects/data/cattaneo2.dta"
# Load the data (use pandas for this)
df = pd.read_stata(data_path,convert_categoricals=False)
# Print first few rows of the data
print(df.head())

# Step 2: Rename variables (using pandas)
df = df.rename(columns={'mbsmoke': 'treat'})
# Display the first few rows of the data
print(df.head())

# Define the covariates and treatment variable
covariates = ['bweight', 'mmarried', 'mage']
treatment = 'treat'
outcome = 'bweight'  # Example outcome variable to estimate ATE

# 1. Standardize the covariates
scaler = StandardScaler()
X = scaler.fit_transform(df[covariates])

# 2. Fit a logistic regression model to estimate the propensity scores (probability of treatment)
model = LogisticRegression(solver='liblinear')
model.fit(X, df[treatment])

# 3. Get the estimated propensity scores
propensity_scores = model.predict_proba(X)[:, 1]  # Probability of receiving treatment (1)

# 4. Perform nearest-neighbor matching using sklearn's NearestNeighbors
nn = NearestNeighbors(n_neighbors=1)
nn.fit(X)
distances, indices = nn.kneighbors(X)

# 5. Create a matched dataset using the indices of the matched samples
matched_data = df.iloc[indices.flatten()]

# 6. Assign matched pairs and estimate ATE

# Treated units: those who received the treatment (1)
treated_units = df[df[treatment] == 1]
# Control units: those who did not receive the treatment (0)
control_units = df[df[treatment] == 0]

# Calculate ATE as the difference in means between treated and matched control units
# The matched control units are found in the 'matched_data' DataFrame
matched_controls = matched_data[df[treatment] == 0]

# ATE formula: ATE = (Average outcome for treated units) - (Average outcome for matched control units)
ate = treated_units[outcome].mean() - matched_controls[outcome].mean()

print(f"Estimated ATE: {ate}")



In [ ]:
#Copilot example

import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Load your dataset
data_path = "/Causal_Inference/Treatment_effects/data/cattaneo2.dta"
# Load the data (use pandas for this)
df = pd.read_stata(data_path,convert_categoricals=False)

# Assume 'treatment' is the treatment indicator, 'outcome' is the outcome variable,
# and the rest are covariates
treatment = df['mbsmoke']
outcome = df['bweight']
covariates = df.drop(columns=['mbsmoke', 'bweight'])

# Separate treated and control units
treated = df[df['mbsmoke'] == 1]
control = df[df['mbsmoke'] == 0]

# Fit nearest neighbors model on control units
nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[covariates.columns])

# Find nearest neighbors for treated units
distances, indices = nn.kneighbors(treated[covariates.columns])

# Get matched control outcomes
matched_control_outcomes = control.iloc[indices.flatten()]['bweight'].values

# Calculate the ATE
ate = np.mean(treated['bweight'].values - matched_control_outcomes)
print(f'Average Treatment Effect (ATE): {ate}')

In [ ]:
import datetime
print(f"NNM estimators finished")
print(f"Program completed on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")